# Group review and human oversight

This notebook checks whether changing a field reserved for monitoring changes the deterministic policy route. It is a counterfactual smoke check, not a statistical fairness assessment. The cohort has only 12 synthetic rows.


In [7]:
from pathlib import Path
import sys
import pandas as pd
PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
import loanguard as loan
applications = pd.read_csv(PROJECT / "synthetic_applications.csv")
store = loan.build_or_load_store(rebuild=False, epochs=100)
verifier = loan.build_verifier(store)
results = [loan.screen_application(store, verifier, r) for r in applications.to_dict("records")]
df = applications.merge(pd.DataFrame(results), on=["applicant_id", "product"])
df.groupby("marital_status", dropna=False).agg(
    applications=("applicant_id", "count"),
    eligible_for_underwriting=("routing", lambda x: (x == "POLICY_ELIGIBLE_FOR_UNDERWRITING").sum()),
    rejected_by_screen=("routing", lambda x: (x == "PRE_SCREEN_REJECTED").sum()),
    manual_review=("routing", lambda x: (x == "MANUAL_REVIEW").sum()),
    always_human_review=("human_decision_required", "all"),
)


  GMS entities:  52
  GMS relations: 11
  GMS triples:   53
  Store loaded from N:\Solytics\Loan_Approval\loan_store
  Entities:  52
  Relations: 11
  Triples:   53
  ENM:       18
  Documents: 1
[1] loaded existing store from N:\Solytics\Loan_Approval\loan_store
  Calibrated geodesic      :   106 samples, 2-class, train accuracy=59.4%  τ=1.5194694393873216
  Calibrated tension       :   530 samples, 2-class, train accuracy=94.3%  τ=DEGENERATE (no usable two-cut separation)
[2] tau_plausibility=1.4921 (95% CI [1.2627, 1.6926]), tension_status=degenerate


,applications,eligible_for_underwriting,rejected_by_screen,manual_review,always_human_review
marital_status,,,,,
Divorced,2,1,0,1,True
Married,4,2,0,2,True
Single,5,2,0,3,True
Widowed,1,0,0,1,True


## Counterfactual field invariance

`marital_status` is not passed into `screen_application`. For each row, change only this field and recompute the route. A match shows only that this implementation does not read that field directly. Because the field is not an input to the rule function, a zero-change result is expected and cannot detect proxy effects or establish fairness. DTI is also not used because the synthetic policy defines no DTI cutoff.


In [8]:
counterfactual = applications.copy()
choices = ["single", "married", "divorced", "widowed"]
original_routes = [loan.screen_application(store, verifier, r)["routing"] for r in applications.to_dict("records")]
changed_rows = applications.to_dict("records")
for i, row in enumerate(changed_rows):
    row["marital_status"] = choices[(choices.index(str(row["marital_status"]).lower()) + 1) % len(choices)] if str(row["marital_status"]).lower() in choices else "married"
changed_routes = [loan.screen_application(store, verifier, r)["routing"] for r in changed_rows]
print({"rows": len(original_routes), "route_changes_after_field_edit": sum(a != b for a,b in zip(original_routes, changed_routes))})


{'rows': 12, 'route_changes_after_field_edit': 0}


## Review boundaries

- Group sizes are too small for statistical disparity measures.
- No protected-class labels or representative population data are supplied.
- The pre-screen is a policy consistency check, not a credit-risk model.
- Every route requires a human underwriter; this demo does not approve or decline a loan.
- A production review needs validated policy, data quality controls, documented appeal paths, and appropriate legal and compliance assessment.
